# Quadtree Tile Index

Reads image pair labels, selects the coarsest TIFF pyramid page that still covers the CNN input dimensions at each quadtree depth, and writes one entry per `(pair, depth, x, y)` tile to `quadtree_annotations.json`.

In [8]:
from pathlib import Path
import json
import sys

import numpy as np
import tifffile
from PIL import Image

sys.path.append(str(Path.cwd().parent))

import conf


IMAGE_DIR = conf.IMAGE_DIR
LABELS_PATH = conf.LABELS_PATH
ANNOTATION_PATH = conf.ANNOTATION_PATH

WSI_PAGES = conf.WSI_PAGES
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH
MAX_CROP_DEPTH = conf.MAX_CROP_DEPTH

print(f"System : {conf.SYSTEM_PREFIX}")
print(f"Labels : {LABELS_PATH}")
print(f"Output : {ANNOTATION_PATH}")

System : macos
Labels : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_labels.json
Output : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_quadtree_annotations.json


In [9]:
WHITE_CUTOFF = 0.87
WHITE_AREA_THRESHOLD = 0.5
POOL_SIZE = 16

In [10]:
def is_background_tile(path, pyramid_page_idx, grid, x_idx, y_idx, page_cache):
    """True when > WHITE_AREA_THRESHOLD fraction of POOL_SIZE×POOL_SIZE pooled pixels exceed WHITE_CUTOFF."""
    path = conf.resolve(path)
    key = (str(path), pyramid_page_idx)
    if key not in page_cache:
        with tifffile.TiffFile(path) as slide:
            page_cache[key] = slide.pages[pyramid_page_idx].asarray()
    page = page_cache[key]
    H, W = page.shape[:2]
    tile_w = W // grid
    tile_h = H // grid
    x0, y0 = x_idx * tile_w, y_idx * tile_h
    x1 = W if x_idx == grid - 1 else x0 + tile_w
    y1 = H if y_idx == grid - 1 else y0 + tile_h
    crop = page[y0:y1, x0:x1]
    pil = Image.fromarray(crop.astype(np.uint8) if crop.dtype != np.uint8 else crop).convert("L")
    pooled = np.array(pil.resize((POOL_SIZE, POOL_SIZE), Image.BILINEAR), dtype=np.float32) / 255.0
    return float((pooled > WHITE_CUTOFF).mean()) > WHITE_AREA_THRESHOLD

In [11]:
def matrix_dict_to_list(m):
    return [
        [m["t_00"], m["t_01"], m["t_02"]],
        [m["t_10"], m["t_11"], m["t_12"]],
        [m["t_20"], m["t_21"], m["t_22"]],
    ]


def load_pairs_from_labels(labels_path, image_dir):
    with open(labels_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    pairs = []

    for pair_id, item in enumerate(labels):
        source_id = item["source_image_id"]
        target_id = item["target_image_id"]

        source_path = image_dir / f"{source_id}.data"
        target_path = image_dir / f"{target_id}.data"

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        if not target_path.exists():
            raise FileNotFoundError(target_path)

        pairs.append({
            "pair_id": pair_id,
            "moving_path": conf.image_relpath(source_id),
            "fixed_path": conf.image_relpath(target_id),
            "source_image_id": source_id,
            "target_image_id": target_id,
            "registration_error": item["registration_error"],
            "transformation_matrix": matrix_dict_to_list(item["transformation_matrix"]),
        })

    return pairs

In [12]:
def choose_pair_pyramid_page(fixed_path, moving_path, crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    fixed_path = conf.resolve(fixed_path)
    moving_path = conf.resolve(moving_path)
    grid = 2 ** crop_depth

    with tifffile.TiffFile(fixed_path) as fixed_slide, tifffile.TiffFile(moving_path) as moving_slide:
        candidates = []

        for page_idx in WSI_PAGES:
            fixed_h, fixed_w = fixed_slide.pages[page_idx].shape[:2]
            moving_h, moving_w = moving_slide.pages[page_idx].shape[:2]

            fixed_tile_h = fixed_h // grid
            fixed_tile_w = fixed_w // grid

            moving_tile_h = moving_h // grid
            moving_tile_w = moving_w // grid

            min_tile_h = min(fixed_tile_h, moving_tile_h)
            min_tile_w = min(fixed_tile_w, moving_tile_w)

            if min_tile_h >= input_height and min_tile_w >= input_width:
                candidates.append((page_idx, min_tile_h, min_tile_w))

    if not candidates:
        return None

    return candidates[-1]

In [13]:
def build_quadtree_index(pairs, max_crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    tile_jobs = []
    page_cache = {}

    for pair in pairs:
        for crop_depth in range(max_crop_depth + 1):
            chosen = choose_pair_pyramid_page(
                fixed_path=pair["fixed_path"],
                moving_path=pair["moving_path"],
                crop_depth=crop_depth,
                input_height=input_height,
                input_width=input_width,
            )

            if chosen is None:
                continue

            pyramid_page_idx, tile_h, tile_w = chosen
            grid = 2 ** crop_depth

            for y_idx in range(grid):
                for x_idx in range(grid):
                    fixed_bg = is_background_tile(
                        pair["fixed_path"], pyramid_page_idx,
                        grid, x_idx, y_idx, page_cache,
                    )
                    moving_bg = is_background_tile(
                        pair["moving_path"], pyramid_page_idx,
                        grid, x_idx, y_idx, page_cache,
                    )
                    tile_jobs.append({
                        "pair_id": pair["pair_id"],
                        "fixed_path": pair["fixed_path"],
                        "moving_path": pair["moving_path"],
                        "source_image_id": pair["source_image_id"],
                        "target_image_id": pair["target_image_id"],
                        "crop_depth": crop_depth,
                        "grid": grid,
                        "x_idx": x_idx,
                        "y_idx": y_idx,
                        "pyramid_page_idx": pyramid_page_idx,
                        "tile_h": tile_h,
                        "tile_w": tile_w,
                        "cnn_input_height": input_height,
                        "cnn_input_width": input_width,
                        "registration_error": pair["registration_error"],
                        "transformation_matrix": pair["transformation_matrix"],
                        "binary_mask_excluded": fixed_bg or moving_bg,
                    })

    return tile_jobs


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [14]:
pairs = load_pairs_from_labels(LABELS_PATH, IMAGE_DIR)

tile_jobs = build_quadtree_index(
    pairs=pairs,
    max_crop_depth=MAX_CROP_DEPTH,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
)

save_json(tile_jobs, ANNOTATION_PATH)

print(f"pairs     : {len(pairs)}")
print(f"tile jobs : {len(tile_jobs)}")
print(f"saved to  : {ANNOTATION_PATH}")

print("\nFirst job:")
print(tile_jobs[0])

print("\nLast job:")
print(tile_jobs[-1])

pairs     : 8
tile jobs : 10920
saved to  : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_quadtree_annotations.json

First job:
{'pair_id': 0, 'fixed_path': '/Users/alexanderhallmann/Desktop/medical-image-registration/data/images/6045.data', 'moving_path': '/Users/alexanderhallmann/Desktop/medical-image-registration/data/images/6036.data', 'source_image_id': 6036, 'target_image_id': 6045, 'crop_depth': 0, 'grid': 1, 'x_idx': 0, 'y_idx': 0, 'pyramid_page_idx': 4, 'tile_h': 1388, 'tile_w': 2054, 'cnn_input_height': 344, 'cnn_input_width': 512, 'registration_error': 104.38675487149722, 'transformation_matrix': [[0.999398059318965, -0.00020282846028934223, -1243.5138093608236], [0.00024455038200216076, 0.9999230769021725, -32.762125098451975], [0.0, 0.0, 1.0]], 'binary_mask_excluded': False}

Last job:
{'pair_id': 7, 'fixed_path': '/Users/alexanderhallmann/Desktop/medical-image-registration/data/images/6051.data', 'moving_path': '/Users/alexanderhallmann/Desktop/me